# Restricted-input method reference
Source: analysis/inter-park/landuse_analysis.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

import os
# Working directory is explicitly configured in the first cell.


In [ ]:
# --- Load GeoJSON ---
gdf = gpd.read_file("gis_files/landuse/landuse_1km_buffer_park.geojson")

# Expected columns: 'layer' (park name), 'LandUse' (category), 'area' (area of polygon)
assert {'layer', 'LandUse', 'area'}.issubset(gdf.columns), "Missing required columns"

In [ ]:
# --- Compute total area per park ---
total_area = gdf.groupby('layer')['area'].sum().rename('total_area')

# Merge total area back
gdf = gdf.merge(total_area, on='layer')

# --- Compute proportion p_i for each LandUse type within each park ---
gdf['p'] = gdf['area'] / gdf['total_area']

gdf.head()

In [ ]:
# --- Compute Shannon entropy per park ---
def shannon_entropy(subdf: pd.DataFrame) -> float:
    p = subdf['p'].values
    p = p[p > 0]
    H = -np.sum(p * np.log(p))
    # normalised by ln(k)
    k = len(p)
    return H / np.log(k) if k > 1 else 0.0

entropy_df = gdf.groupby('layer').apply(shannon_entropy).reset_index()
entropy_df.columns = ['layer', 'Normalized_Shannon_Entropy']


# --- Optional: save result ---
entropy_df.to_csv("landuse_entropy.csv", index=False)
entropy_df

In [ ]:
# calculate the residentice % of each park
residence_per = gdf[gdf['LandUse'] == 'Residential'].groupby('layer')['p'].sum().reset_index()
residence_per.columns = ['layer', 'Residential_Percentage']
residence_per